# Root Zone Water Estimation with Satellite Data — filtering sparse, indirect water information

**Assumed spring wheat · assumed rain-fed (irrigation = 0) · unverified demonstration area**

We estimate one hidden state, total root-zone water storage $W$ in **millimetres**. Daily weather
drives a water balance. Sparse Sentinel-2 NDMI, when genuinely acquired, is converted to an
uncalibrated water proxy and can correct the prediction. Daily propagation bridges image gaps;
filtering combines uncertain dynamics with uncertain indirect information.

There are **no field-measured root-zone observations**. Crop, management, soil and proxy relations
are assumptions. All water estimates are speculative and conditional. Proxy agreement is not
physical validation, even if a curve or uncertainty band looks convincing.

The saved demonstration dataset contains 90 complete weather days and sparse Sentinel-2 observations.
The notebook runs entirely from that offline dataset and does not require credentials or network access.
No simulated observations, synthetic experiment or ground truth fills missing satellite days.

## A student's route through the problem

A rain-fed crop lives on water stored in the soil between rainfall events. Rain replenishes that
store; evaporation, root uptake and drainage reduce it. Weather tells us about rainfall and
atmospheric demand, while satellite images describe vegetation at the surface. Neither directly
measures the amount of water around the roots, and usable images are intermittent.

Our task is to maintain a daily estimate by combining a simple water-balance model with occasional
satellite information. We predict storage first, then use an accepted observation to adjust that
prediction. This is a state-estimation problem: identify the inputs, define the hidden state,
construct its dynamics, and introduce filtering.

### Inputs and their roles

| Input | What it tells us | Role in this notebook |
|---|---|---|
| Open-Meteo precipitation | Water arriving as rain | Adds water to the daily balance |
| Open-Meteo reference ET0 | Atmospheric demand from a reference surface | Helps estimate crop water loss |
| Sentinel-2 NDVI | Vegetation greenness | Context, or optional crop-coefficient input |
| Sentinel-2 NDMI | Moisture-sensitive vegetation signal | Indirect observation after an assumed conversion |

The saved example covers 90 model days, not 90 satellite observations. Later cells report actual
coverage and accepted observation dates.

## Soil storage in one compartment

We replace a spatially varying root system with one compartment of fixed effective depth $Z_r$.
Its state is total root-zone water storage $W$ in millimetres. One millimetre over one square
metre is one litre. Saturation, field capacity and wilting point give that state a physical scale:

$$W_{fc}=1000Z_r\theta_{fc},\qquad W_{wp}=1000Z_r\theta_{wp},\qquad
W_{sat}=1000Z_r\theta_{sat}.$$

The total available water is $TAW=W_{fc}-W_{wp}$. Plant-available water and depletion are

$$A=\operatorname{clip}(W-W_{wp},0,TAW),\qquad
D_r=\operatorname{clip}(W_{fc}-W,0,TAW).$$

These relationships follow the [FAO-56 root-zone framework](https://www.fao.org/4/X0490E/x0490e0e.htm).
They do not claim that the unverified demonstration rectangle has these measured soil properties.

## Predicting the next storage value

The water-balance principle is **next storage = current storage + gains − losses**. Rain is the
only external input in the rain-fed example. Potential crop demand is $K_cET_0$; actual use is
limited by the stress coefficient

$$K_s(W)=\operatorname{clip}\left(\frac{W-W_{wp}}{(1-p)TAW},0,1\right).$$

The implemented daily sequence is:

1. Add effective rain $eP_{rain}$ and record overflow above saturation.
2. Calculate evapotranspiration from wetted storage, limited by extractable water.
3. Drain fraction $d$ of remaining water above field capacity.

With $W_b=\min(W+eP_{rain},W_{sat})$:

$$ET_a=\min\left(K_s(W_b)K_cET_0,\max(W_b-W_{wp},0)\right),$$
$$D=d\max(W_b-ET_a-W_{fc},0),\qquad W'=W_b-ET_a-D.$$

Ineffective rain and overflow are recorded separately. This is a daily single-compartment
approximation; rainfall timing, layers, capillary rise, detailed infiltration and snow processes
are outside the model.

## From satellite signal to an observation

Sentinel-2 compares reflected light in different bands:

$$NDVI=\frac{B08-B04}{B08+B04+\epsilon},\qquad
NDMI=\frac{B8A-B11}{B8A+B11+\epsilon}.$$

The data preparation averages valid per-pixel indices. NDVI provides vegetation context and NDMI
is moisture-sensitive, but neither directly measures root-zone storage. We therefore assume the
editable proxy relationship

$$z_t=W_{wp}+TAW\operatorname{clip}\left(\frac{NDMI-m_{dry}}{m_{wet}-m_{dry}},0,1\right).$$

The filter treats this converted value as an indirect observation of storage, not as a field
measurement. The conversion is speculative and does not remove systematic canopy, soil or parcel
bias.

## Area, data and declared timing

The configured rectangle is near Skopje, 21.43–21.44° E and 41.99–42.00° N. It is not an
authoritative agricultural parcel. Existing seeded farm points/hectares do not establish a wheat
field. We keep its exact boundary and label `geometry_status=unverified`, `crop_status=assumed`,
`irrigation_status=assumed_rainfed`. Satellite land-cover suitability remains unresolved.

Study window: **2025-05-01 through 2025-07-29**, inclusive, UTC. An assumed spring-wheat planting
on 2025-04-01 uses 20/30/60/40-day initial/development/mid/late stages, transferred from the
[FAO56 small-grain calendar example](https://www.fao.org/4/X0490E/x0490e0b.htm).
This choice avoids modeling winter dormancy. Snowfall/freezing weather is rejected; no snowpack
or frozen-soil processes are implemented. The absence of snowfall during this window does not
prove the absence of antecedent snowmelt.

A day is [00:00 UTC, next 00:00 UTC). Predict using complete-day retrospective weather, then
assign an actual overpass proxy to day-end. That alignment is an approximation. It is not real-time
forecasting. Newly observed NDVI may influence only subsequent daily intervals.

## Agriculture for filtering students

Soil **retention** describes water held in pores against gravity. **Field capacity** is the water
remaining after rapid gravitational drainage slows; it is not complete saturation. At the
**permanent wilting point**, water remains in the soil but plants cannot readily extract it.
**Infiltration** carries surface water into the soil. Roots take water up; much leaves the canopy
as **transpiration**, while wet soil loses water directly as **evaporation**. **Drainage** moves
excess water below the represented root zone. One parcel-average compartment omits spatial
differences, layers, preferential flow, rooting variability and lateral exchanges.

One millimetre over one square metre is one litre. Distinguish:

$$W_{fc}=1000Z_r\theta_{fc},\quad W_{wp}=1000Z_r\theta_{wp},\quad TAW=W_{fc}-W_{wp}$$
$$A=\operatorname{clip}(W-W_{wp},0,TAW),\quad D_r=\operatorname{clip}(W_{fc}-W,0,TAW),\quad RAW=p\,TAW.$$

$W$ is total water; $A$ is plant-available water; $D_r$ is depletion from field capacity.
$A=TAW-D_r$ within the available-water range. Volumetric contents $\theta$ have units m³/m³,
root depth $Z_r$ is metres, and storages are mm. Root depth is fixed throughout each run.
Separate root-depth sensitivity runs recompute all capacities, proxy scaling and storage noise.
See [FAO56 Chapter 8](https://www.fao.org/4/X0490E/x0490e0e.htm).

| Symbol | Code | Unit |
|---|---|---|
| $W$ | `W_prior_mm`, `W_posterior_mm` | mm |
| $Z_r$ | `Parameters.root_depth_m` | m |
| $\theta_{fc},\theta_{wp}$ | `theta_fc`, `theta_wp` | m³/m³ |
| $W_{fc},W_{wp},TAW,RAW$ | `p.fc`, `p.wp`, `p.taw`, `p.raw` | mm |
| $K_s,K_c$ | `stress()`, daily `kc` | dimensionless |
| $P, R_t, Q_t$ | `P_*_mm2`, `process_covariance_Rt_mm2`, `measurement_covariance_Qt_mm2` | mm², scalar (1×1) |

If percentages are wanted, $100A/TAW$ is **relative available water**; $W/(1000Z_r)$ is
volumetric water content. Neither is an unspecified dashboard “moisture percentage.”

## Current water-balance model

Our main teaching model uses fixed-root-depth soil capacities and the readily-available-water plateau:

$$K_s(W)=\operatorname{clip}\left(\frac{W-W_{wp}}{(1-p)TAW},0,1\right),\qquad 0<p<1.$$

Daily wetting $W_{wet}=W+eP$ precedes losses. Record ineffective precipitation $(1-e)P$;
record overflow $O=\max(W_{wet}-W_{sat},0)$ and subtract it. Then:

$$ET_a=\min(K_s(W_{wet}-O)K_cET_0,\max(W_{wet}-O-W_{wp},0))$$
$$D=d\max(W_{wet}-O-ET_a-W_{fc},0),\qquad W'=W_{wet}-O-ET_a-D.$$

Thus $W'-W=P-(1-e)P-O-ET_a-D$, with irrigation exactly zero. No unexplained clipping discards
water. Saturation comes from an explicitly assumed volumetric porosity. This ordering is a daily
approximation; rainfall timing within the day is lost. Interception, capillary rise, infiltration-rate
limits, snow and layers are omitted. Effectiveness defaults to 1, up to overflow.

The single-coefficient approximation $ET_a=K_sK_cET_0$ treats combined evaporation/transpiration
with one stress factor. It is not a full plant model and may suppress soil evaporation too strongly.
Kc describes potential crop use; assumed rain-fed conditions affect actual use through storage and Ks.

## Exact observation construction and why EKF/UKF

The satellite inputs use these bands and aggregation:

$$NDVI=\frac{B08-B04}{B08+B04+\epsilon},\qquad NDMI=\frac{B8A-B11}{B8A+B11+\epsilon}.$$

We average valid per-pixel indices, rather than forming an index from average reflectance.
[Sentinel Hub's NDMI reference](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/ndmi/)
describes the moisture-sensitive canopy index. That does not calibrate root-zone water.

One shared heuristic maps the index to storage:

$$z=W_{proxy}=W_{wp}+TAW\,\operatorname{clip}\left(\frac{NDMI-m_{dry}}{m_{wet}-m_{dry}},0,1\right),$$
$$z=h(W)+v,\qquad h(W)=W,\quad H=1.$$

Default endpoints $m_{dry}=-.10$, $m_{wet}=.45$ define the configured teaching heuristic, with
no demonstrated calibration/coefficient provenance for wheat. The alternate endpoint pair -.20/.60
is available only as a sensitivity assumption.
Both remain assumptions. Clipping censors information; it does not remove uncertainty.

The correction is **linear**. Thresholds and switching in the process cause global nonlinearity;
within affine regimes EKF and UKF can be close. We do not invent a canopy observation function,
predicted-NDVI curve, or promise a UKF advantage. NDVI is vegetation context in calendar mode;
optional NDVI Kc mode treats it as forcing, never a second measurement. Valid NDMI can
correct without NDVI; valid NDVI alone cannot correct W. Same-image index errors may be correlated,
an approximation not represented by the scalar independent-noise filter.

## Configuration and uncertainty

All editable assumptions live in `config.json`, with separate crop/soil references. Calendar Kc is
constant in initial/mid stages and linear in development/late stages. The spring-wheat initial Kc
is .30, and a fixed effective depth is a teaching simplification even during development.

Initial storage at study-start is the available-range midpoint; $P_0=TAW^2/12$ approximates a
uniform prior by a Gaussian. Initialization precedes the first possible image, avoiding artificial
first-proxy zero residuals. Standard deviations in config are converted to mm then squared:

$$\sigma_{W,index}=\frac{TAW}{m_{wet}-m_{dry}}\sigma_{NDMI},\quad
Q_t=\sigma_{W,index}^2+\sigma_{proxy-model}^2.$$

NDMI observation floor .05; proxy discrepancy .20*TAW; process discrepancy .02*TAW per day.
Larger pixel spatial dispersion can replace the NDMI floor as a conservative heuristic, not an
estimated independent-pixel error of the mean. Missing dispersion retains both uncertainty terms.
At clipped endpoints $Q_t$ doubles; the interior Gaussian propagation remains approximate.

$$R_t=\sigma_{process}^2+(\partial f/\partial P)^2\sigma_P^2+
(\partial f/\partial ET_0)^2\sigma_{ET_0}^2.$$

Optional weather terms use 1 mm precipitation and .5 mm ET0 standard deviations, with local
process sensitivities. Errors are assumed independent; the process allowance represents residual
dynamics discrepancy separately. These values are teaching choices, not calibrated errors.
All Q and R components are explicit configurable uncertainty terms; spatial standard deviation is
not treated as proxy-model uncertainty by itself.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from root_zone_water.config import load_config, parameters
from root_zone_water.data import load_data
from root_zone_water.model import budget
from root_zone_water.proxy import from_ndmi
from root_zone_water.filters import FILTERS
from root_zone_water.runner import input_history, run, compare, diagnostics, sensitivity
from root_zone_water import plotting
config = load_config(ROOT / 'config_test.json')
p, crop, soil = parameters(config)
assert config['crop'] == 'wheat' and config['irrigation_mode'] == 'assumed_rainfed'
display(pd.Series(config))
display(pd.read_csv(ROOT / 'references/crops.csv'))
display(pd.read_csv(ROOT / 'references/soils.csv'))
display(pd.Series({'W_wp_mm':p.wp,'W_fc_mm':p.fc,'TAW_mm':p.taw,'RAW_mm':p.raw,
                   'saturation_mm':p.saturation,'initial_W_mm':p.wp+config['initial_available_fraction']*p.taw,
                   'initial_std_mm':config['initial_std_taw_fraction']*p.taw,
                   'process_discrepancy_std_mm':config['process_std_taw_fraction']*p.taw,
                   'proxy_model_std_mm':config['proxy_model_std_taw_fraction']*p.taw}))

crop                                                                        wheat
crop_status                                                               assumed
soil                                                                         silt
irrigation_mode                                                   assumed_rainfed
irrigation_status                                                 assumed_rainfed
geometry_status                                                        unverified
geometry                        {'type': 'Polygon', 'coordinates': [[[21.43, 4...
start_date                                                             2026-05-01
end_date                                                               2026-07-29
timezone                                                                      UTC
planting_date                                                          2026-04-01
kc_mode                                                                  calendar
ndvi_max_age_day

,crop,subtype,kc_initial,kc_mid,kc_end,initial_days,development_days,mid_days,late_days,root_depth_m,root_depth_reference_range_m,p,default_ndvi,source,notes
0,wheat,spring wheat,0.3,1.15,0.25,20,30,60,40,1.4,1.0-1.5,0.55,0.60,https://www.fao.org/4/X0490E/x0490e0b.htm;http...,FAO56 tables 11/12/22; April Mediterranean sma...
1,maize,field grain maize,0.3,1.20,0.60,30,40,50,30,1.0,1.0-1.7,0.55,0.68,https://www.fao.org/4/X0490E/x0490e0b.htm;http...,FAO56 Kc and depth/p; stage days are teaching ...
2,tomato,field tomato,0.6,1.15,0.80,30,40,40,25,1.0,0.7-1.5,0.40,0.72,https://www.fao.org/4/X0490E/x0490e0b.htm;http...,FAO56 Kc and depth/p; stage days are teaching ...


,soil,theta_fc_m3m3,theta_wp_m3m3,theta_sat_m3m3,drainage_fraction_day,source,notes
0,silt,0.32,0.15,0.46,0.1,https://www.fao.org/4/X0490E/x0490e0e.htm,FC/WP reference example36 table19 selections; ...
1,loamy_sand,0.15,0.06,0.40,0.2,https://www.fao.org/4/X0490E/x0490e0e.htm,FC/WP reference example36 selections; saturati...


W_wp_mm                       210.000000
W_fc_mm                       448.000000
TAW_mm                        238.000000
RAW_mm                        130.900000
saturation_mm                 644.000000
initial_W_mm                  329.000000
initial_std_mm                 68.704682
process_discrepancy_std_mm     11.900000
proxy_model_std_mm             11.900000
dtype: float64

## Inspect acquired data before estimating

Cached mode verifies file checksums and reconstructs compact tables from raw API responses.
It requires no network or credentials. Missing required weather causes an error; missing satellite
acquisition is clearly reported in the manifest and remains missing in the daily table.
Full provenance, raw-response schemas and policy are in `docs/DATASET.md`.

In [2]:
weather, satellite, manifest = load_data(config, mode='cached')
display(pd.Series({'dataset_status':manifest['status'], 'weather_days':len(weather),
                   'satellite_rows':len(satellite),
                   'accepted_ndmi_days':int(satellite['ndmi_accepted'].sum()),
                   'retrieved_at_utc':manifest['retrieved_at_utc'], 'blockers':manifest['errors']}))
display(manifest.get('weather_returned_metadata'))
display(weather.head())
display(satellite.head())
plotting.area(config, satellite)
plt.show()

dataset_status                                complete
weather_days                                        90
satellite_rows                                      45
accepted_ndmi_days                                  23
retrieved_at_utc      2026-09-11T11:39:22.338567+00:00
blockers                                            []
dtype: object

{'latitude': 42.0,
 'longitude': 21.5,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 240.0,
 'daily_units': {'time': 'iso8601',
  'precipitation_sum': 'mm',
  'rain_sum': 'mm',
  'snowfall_sum': 'cm',
  'et0_fao_evapotranspiration': 'mm',
  'temperature_2m_min': '°C'}}

,date,precipitation_mm,rain_mm,snowfall_cm,et0_mm,temperature_min_c
0,2026-05-01,0.0,0.0,0.0,2.99,3.6
1,2026-05-02,0.0,0.0,0.0,4.11,2.7
2,2026-05-03,0.1,0.1,0.0,4.42,5.1
3,2026-05-04,0.0,0.0,0.0,4.96,3.1
4,2026-05-05,0.0,0.0,0.0,5.15,5.9


,date,product_ids,acquisition_times_utc,acquisition_time_utc,catalog_reconciled,observation_id,ndvi,ndvi_spatial_std,ndvi_sample_count,ndvi_valid_count,...,ndmi_valid_count,ndmi_valid_fraction,ndmi_accepted,vegetation,vegetation_spatial_std,vegetation_sample_count,vegetation_valid_count,vegetation_valid_fraction,vegetation_accepted,rejection_reason
0,2026-05-02,S2C_MSIL2A_20260502T092031_N0512_R093_T34TEM_2...,2026-05-02T09:28:57.015Z,2026-05-02T09:28:57.015Z,True,S2C_MSIL2A_20260502T092031_N0512_R093_T34TEM_2...,0.231236,0.224744,2352,2351,...,2351,0.999575,True,0.156104,0.362954,2352,2351,0.999575,True,
1,2026-05-04,S2A_MSIL2A_20260504T092231_N0512_R093_T34TEM_2...,2026-05-04T09:29:09.511Z,2026-05-04T09:29:09.511Z,True,S2A_MSIL2A_20260504T092231_N0512_R093_T34TEM_2...,0.223356,0.223563,2352,2352,...,2352,1.000000,True,0.164966,0.371150,2352,2352,1.000000,True,
2,2026-05-05,S2C_MSIL2A_20260505T093041_N0512_R136_T34TEM_2...,2026-05-05T09:38:52.964Z,2026-05-05T09:38:52.964Z,True,S2C_MSIL2A_20260505T093041_N0512_R136_T34TEM_2...,0.227708,0.214420,2352,2352,...,2352,1.000000,True,0.150085,0.357155,2352,2352,1.000000,True,
3,2026-05-07,S2A_MSIL2A_20260507T093041_N0512_R136_T34TEM_2...,2026-05-07T09:28:53.113Z;2026-05-07T09:39:04.748Z,,False,S2A_MSIL2A_20260507T093041_N0512_R136_T34TEM_2...,NaN,NaN,2352,0,...,0,0.000000,False,NaN,NaN,2352,0,0.000000,False,unreconciled_or_multiple_acquisitions
4,2026-05-10,S2B_MSIL2A_20260510T093029_N0512_R136_T34TEM_2...,2026-05-10T09:38:49.268Z,2026-05-10T09:38:49.268Z,True,S2B_MSIL2A_20260510T093029_N0512_R136_T34TEM_2...,NaN,NaN,2352,0,...,0,0.000000,False,NaN,NaN,2352,0,0.000000,False,insufficient_valid_ndmi


Satellite requests use the actual polygon projected to UTM 34N (EPSG:32634), 20 m grid,
nearest resampling, and complete P1D bins ending at **2025-07-30T00:00:00Z**. No-data, SCL
0/1/3/8/9/10/11 and invalid ratio inputs are masked. Remaining SCL classes include water and
bare/nonvegetated land; this is not a wheat mask. At least 100 valid samples and .50 valid fraction
are required. Pixel counts and independence are different questions. `leastCC` alone is insufficient.

Catalog IDs/times are retained. A day with multiple distinct acquisition times is conservatively
rejected; an unambiguous day can contribute one correction. NDVI and NDMI quality are independent.
No forward-filled corrections, guessed acquisition times, or dashboard observations enter this run.

## Common filter interface: visible predict/update

The next cell demonstrates one **real-weather** interval using the selected algorithm; it does not
generate observations. Every comparison below starts afresh. With $G_t=\partial g/\partial x$:

$$\bar{\mu}_t=g(u_t,\mu_{t-1}),\quad \bar{\Sigma}_t=G_t\Sigma_{t-1}G_t^T+R_t,\quad
\nu_t=z_t-h(\bar{\mu}_t),\quad S_t=H_t\bar{\Sigma}_tH_t^T+Q_t,$$
$$K_t=\bar{\Sigma}_tH_t^TS_t^{-1},\quad \mu_t=\bar{\mu}_t+K_t\nu_t,\quad
\Sigma_t=(1-K_tH_t)\bar{\Sigma}_t.$$

For this scalar identity observation, $h(W)=W$ and $H_t=1$, so
$K_t=\bar{\Sigma}_t/(\bar{\Sigma}_t+Q_t)$. The runner also includes optional numerical
derivatives of the weather-forcing response for the $R_t$ calculation; these are not the EKF state
Jacobian. The EKF uses the analytical water-balance Jacobian, with documented branch conventions.
UKF predicts using three sigma points (alpha=1, beta=2, kappa=0). Its linear measurement update
shares the same interface. Out-of-range sigma points are clipped and counted; constrained means
retain variance plus squared projection displacement. These safeguards approximate constrained
distributions; conditional Gaussian bands can extend outside physical bounds.

The state is root-zone storage $W_t$ in millimetres. Sentinel-2 does not directly measure it.
Raw NDMI is converted first into a speculative storage proxy $z_t$, then the assumed observation model is

$$z_t=W_t+\delta_t,\qquad \delta_t\sim N(0,Q_t),\qquad h(W_t)=W_t,\quad H_t=\frac{\partial h}{\partial W_t}=1.$$

The identity is an assumption about the proxy, not a consequence of matching units. A raw-NDMI
observation model would need a forward model predicting NDMI from storage; this project does not use one.

In [3]:
from inspect import getsource
print(getsource(run))

def run(weather,satellite,c,filter_name=None):
    p,_,_ = parameters(c)
    name = filter_name or c['filter']
    f = FILTERS[name](p.wp+c['initial_available_fraction']*p.taw,
                      (c['initial_std_taw_fraction']*p.taw)**2,(0,p.saturation))
    records = []
    for row in input_history(weather,satellite,c):
        before,before_p = f.result()
        # In a validated snow-free interval total precipitation includes liquid showers too.
        rain,et0,kc = row['precipitation_mm'],row['et0_mm'],row['kc']
        transition = lambda w: budget(w,rain,et0,kc,p)[0]
        physical,flux = budget(before,rain,et0,kc,p)
        # Process covariance R_t: model discrepancy plus optional weather uncertainty.
        process_covariance = (c['process_std_taw_fraction']*p.taw)**2
        df_dp = df_det = 0.0
        if c['weather_uncertainty']:
            df_dp = finite_difference_derivative(lambda v:budget(before,v,et0,kc,p)[0],rain,p.taw,0)
            df_det = finite_difference_

In [4]:
results = compare(weather, satellite, config)  # runs the same daily pipeline for each filter
history = input_history(weather, satellite, config)
first = history[0]
selected = FILTERS[config['filter']](p.wp+config['initial_available_fraction']*p.taw,
                              (config['initial_std_taw_fraction']*p.taw)**2, (0,p.saturation))
transition = lambda W: budget(W,first['precipitation_mm'],first['et0_mm'],first['kc'],p)[0]
selected.predict(transition, process_covariance=results[config['filter']].iloc[0]['process_covariance_Rt_mm2'],
                scale=p.taw, state_jacobian=results[config['filter']].iloc[0]['jacobian'])
obs = first['observation']
if obs.get('ndmi_accepted', False) and config['filter'] != 'open_loop':
    measurement = from_ndmi(obs['ndmi'],obs.get('ndmi_spatial_std'),p,config)
    selected.update(observation=measurement['proxy_mm'],
                    measurement_covariance=measurement['measurement_covariance_Qt_mm2'])
else:
    print('First interval: prediction only (no accepted real NDMI correction).')
display(pd.Series(dict(zip(['mean_mm','variance_mm2'],selected.result()))))
display(results[config['filter']].head())
accepted = results[config['filter']].query('ndmi_accepted == True').copy()
display(accepted[['date','W_prior_mm','W_posterior_mm','proxy_mm','P_prior_mm2',
                  'process_covariance_Rt_mm2','measurement_covariance_Qt_mm2',
                  'innovation_mm','gain','assimilation_adjustment_mm',
                  'posterior_bound_adjustment_mm']])

First interval: prediction only (no accepted real NDMI correction).


mean_mm          327.255833
variance_mm2    4863.028403
dtype: float64

,date,precipitation_mm,rain_mm,snowfall_cm,et0_mm,temperature_min_c,kc,stage,kc_source,ndvi_input_age_days,...,df_det0,ndmi_accepted,updated,rejected,rejection_reason,observation_id,assimilation_adjustment_mm,distribution_prediction_adjustment_mm,sigma_index_water_mm,sigma_proxy_model_mm
0,2026-05-01,0.0,0.0,0.0,2.99,3.6,0.583333,development,calendar,NaN,...,-0.583333,False,False,False,no_observation,,0.000000,0.0,NaN,NaN
1,2026-05-02,0.0,0.0,0.0,4.11,2.7,0.611667,development,calendar,NaN,...,-0.611667,True,True,False,,S2C_MSIL2A_20260502T092031_N0512_R093_T34TEM_2...,-38.340579,0.0,47.519271,11.9
2,2026-05-03,0.1,0.1,0.0,4.42,5.1,0.640000,development,calendar,0.604896,...,-0.457151,False,False,False,no_observation,,0.000000,0.0,NaN,NaN
3,2026-05-04,0.0,0.0,0.0,4.96,3.1,0.668333,development,calendar,1.604896,...,-0.464780,True,True,False,,S2A_MSIL2A_20260504T092231_N0512_R093_T34TEM_2...,-7.137625,0.0,48.459490,11.9
4,2026-05-05,0.0,0.0,0.0,5.15,5.9,0.696667,development,calendar,0.604751,...,-0.423059,True,True,False,,S2C_MSIL2A_20260505T093041_N0512_R136_T34TEM_2...,-3.326741,0.0,45.965404,11.9


,date,W_prior_mm,W_posterior_mm,proxy_mm,P_prior_mm2,process_covariance_Rt_mm2,measurement_covariance_Qt_mm2,innovation_mm,gain,assimilation_adjustment_mm,posterior_bound_adjustment_mm
1,2026-05-02,324.741883,286.401305,268.021266,5005.731937,142.703534,2399.691157,-56.720617,0.675955,-38.340579,0.0
3,2026-05-04,282.175390,275.037765,264.707153,1720.343753,142.603060,2489.932138,-17.468237,0.408606,-7.137625,0.0
4,2026-05-05,272.859010,269.532269,262.670300,1092.966151,142.588867,2254.428369,-10.188710,0.326512,-3.326741,0.0
13,2026-05-14,265.234532,264.698941,263.973228,1457.318922,142.609072,1974.630562,-1.261304,0.424633,-0.535591,0.0
19,2026-05-20,268.381581,268.233750,267.992305,1303.021247,142.629429,2128.148770,-0.389276,0.379760,-0.147831,0.0
24,2026-05-25,255.329957,256.993355,260.511528,1053.299967,142.558353,2227.784485,5.181571,0.321022,1.663398,0.0
29,2026-05-30,244.504665,248.271670,256.673847,925.267724,142.523354,2063.778132,12.169182,0.309553,3.767005,0.0
31,2026-06-01,246.074031,249.980367,260.399096,779.607393,142.545844,2079.318645,14.325066,0.272692,3.906337,0.0
39,2026-06-09,240.385207,244.894410,253.029641,1020.218592,142.521338,1840.616791,12.644434,0.356616,4.509203,0.0
41,2026-06-11,245.055907,247.754900,254.681733,788.381606,142.540408,2023.342707,9.625826,0.280391,2.698993,0.0


## Genuine-input comparison and observation gaps

Weather, calendar, NDVI policy and proxy map are identical for all algorithms. In calendar mode
NDVI is context only. In NDVI mode the last valid past image can drive Kc for 15 days before calendar
fallback. Holding a forcing value never creates a fresh observation. No irrigation recommendation
is fed back into the model.

The current weather-only comparison exercises propagation. It cannot evaluate assimilation or
establish an algorithm ranking. After a genuine satellite cache is captured, these same cells display
the actual sparse indices, heuristic proxies and pre-correction innovations automatically.

In [5]:
plotting.inputs(weather,satellite,results['ekf']).show(config=plotting.PLOTLY_CONFIG)
comparison_figure = plotting.states(results,p)
comparison_figure.show(config=plotting.PLOTLY_CONFIG)
plotting.export_html(comparison_figure, ROOT / 'outputs/filter-comparison.html')
plotting.innovations(results).show(config=plotting.PLOTLY_CONFIG)
display(diagnostics(results))
for result in results.values():
    assert len(result) == len(weather)
    assert result['irrigation_mm'].eq(0).all()
    assert result['balance_error_mm'].abs().max() < 1e-8
    assert np.isfinite(result['P_posterior_mm2']).all()
    assert result['P_posterior_mm2'].ge(0).all()

,filter,proxy_pairs,prior_proxy_MAE_mm,prior_proxy_RMSE_mm,corrections,max_physical_balance_error_mm
0,open_loop,23,28.051702,31.815568,0,0.0
1,ekf,23,15.209541,18.548203,23,0.0
2,ukf,23,15.387287,18.805598,23,0.0


The actual observation-space graph compares storage in mm with **NDMI-derived water proxy
(heuristic), mm**. Prior and posterior are shown separately; raw dimensionless indices have their
own plots. $\mu_t-\bar{\mu}_t$ is a **measurement adjustment**, not observed rain or irrigation.
UKF's expectation of a nonlinear transition can also differ from the trajectory at its mean;
`distribution_prediction_adjustment_mm` records that difference separately. Conservation applies
to `physical_prediction_mm` and its named fluxes, not to assimilated knowledge changes.

Pre-correction innovations are $\nu_t=z_t-h(\bar{\mu}_t)$ with $S_t=\bar{\Sigma}_t+Q_t$. Any MAE/RMSE above means prior disagreement
with the heuristic proxy (mm); it is not root-zone field accuracy. There is no posterior-proxy
performance claim, automatic first-proxy initialization success or residual-derived “actual irrigation.”

## Small sensitivity comparisons on the same genuine inputs

Change process noise, proxy discrepancy, initial mean/variance, proxy endpoint pair and root depth
one at a time. Soil/reference thresholds, proxy mapping and storage-scaled uncertainties recompute
for each fixed-depth run. No synthetic experiment is used. Without satellite observations, R and
endpoint changes have no influence; overlapping curves correctly reflect that absence.

In [6]:
sensitivity_runs = sensitivity(weather,satellite,config)
plotting.sensitivities(sensitivity_runs).show(config=plotting.PLOTLY_CONFIG)
display(pd.DataFrame([{'setting':label, 'final_W_mm':r.iloc[-1]['W_posterior_mm'],
                       'final_std_mm':np.sqrt(r.iloc[-1]['P_posterior_mm2']),
                       'corrections':int(r['updated'].sum())} for label,r in sensitivity_runs.items()]))

,setting,final_W_mm,final_std_mm,corrections
0,baseline,243.871120,25.673362,23
1,Q variance x4,249.844265,34.252619,23
2,R larger,242.495108,26.921350,23
3,initial mean wetter,243.871158,25.673362,23
4,initial variance x4,243.871094,25.673362,23
5,older proxy endpoints,260.051817,21.659185,23
6,root depth x0.75,180.638115,18.386388,23


## Interpretation, limitations and extension exercises

Unverified land use/crop, assumed irrigation, heuristic canopy-to-root-zone mapping, fixed reference
soil/root values, daily overpass alignment and noise assumptions dominate interpretation. Wide
bands do not remove systematic bias. Kc describes potential use; water stress limits actual modeled
use. Weather is a reanalysis grid, not a field sensor. Satellite gaps remain visible in reports.

1. After acquiring genuine scenes using `docs/API_SETUP.md`, inspect SCL vegetation context. If it
   indicates predominantly nonvegetated cover, report this rectangle as unsuitable for representative
   crop interpretation. Replace it only with an explicitly verified parcel and update provenance.
2. Compare calendar and causal NDVI Kc modes on the same real scenes. Discuss correlated errors
   between vegetation forcing and moisture proxy, and the consequences of the 15-day age limit.
3. Inspect where the model crosses stress/drainage thresholds and whether EKF/UKF differences
   concentrate there. No algorithm superiority is assumed.
4. If sufficient real images become available, withhold whole images from every dependent path,
   including NDVI Kc inputs, before assessing gap behavior. Separate tuning from evaluation.
   Agreement with withheld proxies still cannot validate absolute storage.
5. Add a filter through `docs/EXTENDING_FILTERS.md`; keep the same runner, scalar proxy and timing.
   Field-measured root-zone water is future work, not an available target dataset.

Relevant sources: [FAO56 crop coefficients](https://www.fao.org/4/X0490E/x0490e0b.htm),
[FAO56 water accounting/stress](https://www.fao.org/4/X0490E/x0490e0e.htm),
[Open-Meteo historical weather](https://open-meteo.com/en/docs/historical-weather-api),
[CDSE Statistical API](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Statistical.html).

In [7]:
config["ndmi_obs_std"]

0.05